# Đề kiểm tra thực hành 1 - Solution

**Phân tích dữ liệu Bayesian - IUH**

Bài thực hành này ngắn hơn các lab chuẩn, nhưng nó gom lại khá nhiều mảnh quan trọng của Bayesian statistics: xác suất toàn phần, tư duy posterior, công thức liên hợp chuẩn, và cách biểu diễn một tiên nghiệm hỗn hợp. Vì vậy nếu chỉ đưa đáp số ngắn, người học sẽ khó thấy được sợi dây chung giữa bốn bài.

Notebook này được viết lại theo phong cách của các solution notebook khác trong repo: mỗi bài đều có phần ý nghĩa, công thức cốt lõi, lời giải số, và một đoạn diễn giải ngắn để người học thấy dữ liệu đã làm thay đổi lập luận như thế nào.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)


## 1. Thiết lập chung cho đề cá nhân hóa

Hai bài trong đề phụ thuộc MSSV. Để notebook vẫn chạy được end-to-end, ta khóa một MSSV minh họa mặc định là `20123671`, đúng theo ví dụ xuất hiện trong đề.

- Với **Bài 2**, ta dùng chữ số nhỏ nhất khác `0` và chữ số lớn nhất để tạo $y = a/b$.
- Với **Bài 4**, ta dùng chữ số gần cuối và chữ số cuối.

Nếu muốn đổi sang MSSV của mình, chỉ cần sửa một biến trong cell dưới đây; toàn bộ phần còn lại sẽ tự cập nhật.


In [ ]:
student_id = "20123671"  # Thay bằng MSSV của bạn nếu cần

digits = [int(ch) for ch in student_id if ch.isdigit()]
if not digits:
    raise ValueError("student_id must contain at least one digit")
if len(digits) < 2:
    raise ValueError("student_id must contain at least two digits for Bai 4")

smallest_nonzero = min(d for d in digits if d != 0)
largest_digit = max(digits)
second_last_digit = digits[-2]
last_digit = digits[-1]

print(f"MSSV dang dung: {student_id}")
print(f"Bai 2 -> a = {smallest_nonzero}, b = {largest_digit}, y = a/b = {smallest_nonzero/largest_digit:.6f}")
print(f"Bai 4 -> a = {second_last_digit}, b = {last_digit}")


## 2. Bài 1 - Xác suất toàn phần trong một bối cảnh quen thuộc

**Đề bài.** Tỷ lệ sinh viên IUH theo ngành CNTT là $20\text{%}$. Trong số sinh viên CNTT thì có $70\text{%}$ bị tật về mắt, còn trong nhóm không học CNTT thì tỷ lệ đó là $15\text{%}$. Hỏi nếu gặp ngẫu nhiên một sinh viên thì xác suất sinh viên đó bị tật về mắt là bao nhiêu?

### Ý nghĩa của bài tập

Đây là một bài mở đầu rất điển hình cho tư duy Bayesian: ta chưa đi ngược từ dữ liệu về nguyên nhân ẩn, nhưng ta đã phải kết hợp nhiều nguồn thông tin điều kiện để ra một xác suất tổng quát hơn. Nói cách khác, đây là bước “gom các kịch bản lại” trước khi học đến bước “cập nhật ngược” bằng Bayes rule.

Về mặt trực giác, bài này nhắc ta rằng một tỷ lệ tổng thể không thể đọc trực tiếp từ một nhóm con duy nhất. Muốn biết xác suất một sinh viên bất kỳ bị tật về mắt, ta phải cân theo tỷ trọng của từng nhóm trong toàn trường.


### Công thức cốt lõi và cách đọc

Gọi:

- $A$: sinh viên học ngành CNTT;
- $M$: sinh viên bị tật về mắt.

Khi đó công thức xác suất toàn phần cho ta:

$$
P(M) = P(M\mid A)P(A) + P(M\mid A^c)P(A^c).
$$

Thay số từ đề bài:

$$
P(M) = 0.7 \cdot 0.2 + 0.15 \cdot 0.8 = 0.14 + 0.12 = 0.26.
$$

Vậy xác suất cần tìm là **$26\%$**.


In [ ]:
p_it = 0.20
p_eye_given_it = 0.70
p_eye_given_non_it = 0.15

p_eye = p_it * p_eye_given_it + (1 - p_it) * p_eye_given_non_it
print(f"Bai 1: P(mat tat ve mat) = {p_eye:.4f} = {100*p_eye:.2f}%")


### Interpretation

Kết quả $26\%$ cho thấy tỷ lệ chung nằm giữa $70\%$ và $15\%$, nhưng không đơn giản là trung bình cộng của hai số đó. Nó nghiêng mạnh về phía $15\%$ vì nhóm không học CNTT chiếm tới $80\%$ tổng số sinh viên.


## 3. Bài 2 - Posterior và ước lượng MAP từ một đề cá nhân hóa

**Đề bài.** Cho prior $X \sim 6x(1-x)$ trên $[0,1]$ và likelihood được cho bởi

$$
f_{Y\mid X}(y\mid x)=x^2+\frac{4y}{3}, \qquad 0 \le x,y \le 1.
$$

Với quan sát $y=\frac{a}{b}$, trong đó $a$ là chữ số nhỏ nhất khác $0$ và $b$ là chữ số lớn nhất trong MSSV, hãy:

- tìm ước lượng MAP;
- vẽ đồ thị mật độ hậu nghiệm.

### Ý nghĩa của bài tập

Bài này đưa người học vào đúng tinh thần Bayesian: có một prior cho tham số ẩn, có dữ liệu quan sát, rồi từ đó xây posterior và tìm một điểm tóm tắt của posterior là MAP. Đây là bước chuyển rất quan trọng từ xác suất có điều kiện thông thường sang suy luận Bayes.

Đồng thời, đây cũng là một ví dụ tốt cho việc giải đề cá nhân hóa theo MSSV. Các con số thay đổi theo từng sinh viên, nhưng cấu trúc posterior và cách lấy MAP không đổi.


### Công thức cốt lõi và cách đọc

Với MSSV hiện tại, ta có:

$$
y_0 = \frac{a}{b}.
$$

Prior của tham số là:

$$
\pi(x)=6x(1-x), \qquad 0 \le x \le 1.
$$

Lưu ý rằng biểu thức trong đề không được chuẩn hóa theo biến $y$, nên ở đây ta xem nó như một **likelihood kernel theo $x$** sau khi đã quan sát $y=y_0$. Khi đó:

$$
p(x\mid y_0) \propto 6x(1-x)\left(x^2+\frac{4y_0}{3}\right), \qquad 0 \le x \le 1.
$$

Hằng số chuẩn hóa là:

$$
Z = \int_0^1 6x(1-x)\left(x^2+\frac{4y_0}{3}\right)dx
  = \frac{3}{10} + \frac{4y_0}{3}.
$$

Do đó:

$$
p(x\mid y_0) = \frac{6x(1-x)\left(x^2+\frac{4y_0}{3}\right)}{\frac{3}{10}+\frac{4y_0}{3}}, \qquad 0 \le x \le 1.
$$

MAP là điểm cực đại của mật độ hậu nghiệm trên $[0,1]$.


In [ ]:
y_obs = smallest_nonzero / largest_digit
normalizing_constant = 3 / 10 + 4 * y_obs / 3

def posterior_kernel_bai2(x, y):
    return 6 * x * (1 - x) * (x**2 + 4 * y / 3)

def posterior_pdf_bai2(x, y):
    return posterior_kernel_bai2(x, y) / (3 / 10 + 4 * y / 3)

result = optimize.minimize_scalar(
    lambda x: -posterior_pdf_bai2(x, y_obs),
    bounds=(0, 1),
    method='bounded'
)
map_estimate_bai2 = result.x

grid_bai2 = np.linspace(0, 1, 500)
density_bai2 = posterior_pdf_bai2(grid_bai2, y_obs)

print(f"Bai 2: y0 = {y_obs:.6f}")
print(f"Bai 2: hang so chuan hoa Z = {normalizing_constant:.6f}")
print(f"Bai 2: MAP xap xi = {map_estimate_bai2:.6f}")

plt.plot(grid_bai2, density_bai2, label='Posterior density')
plt.axvline(map_estimate_bai2, color='crimson', linestyle='--', label=f'MAP = {map_estimate_bai2:.4f}')
plt.title('Bai 2 - Mat do hau nghiem')
plt.xlabel('x')
plt.ylabel('density')
plt.legend()
plt.show()


### Interpretation

Với MSSV minh họa đang dùng, posterior đạt cực đại gần $x \approx 0.7103$. Điều này có nghĩa là sau khi kết hợp prior với thông tin từ quan sát $y_0$, giá trị tham số được posterior ủng hộ mạnh nhất nằm quanh vùng này. Trên đồ thị, đó chính là đỉnh của đường mật độ hậu nghiệm.


## 4. Bài 3 - Normal conjugacy và posterior Gaussian

**Đề bài.** Cho prior

$$
X \sim N(16, 0.5).
$$

Một mẫu ngẫu nhiên gồm 15 quan sát được cho sẵn. Hãy xác định phân phối hậu nghiệm và vẽ prior cùng posterior trên một hệ trục.

### Ý nghĩa của bài tập

Đây là bài mẫu cho liên hợp chuẩn. Một khi prior và likelihood đều Gaussian, posterior cũng là Gaussian. Điều quan trọng cần học ở đây không chỉ là công thức, mà là trực giác: posterior mean là sự cân bằng giữa prior mean và thông tin từ dữ liệu, còn posterior variance sẽ nhỏ hơn prior variance vì sau quan sát ta bớt bất định hơn.

Nhóm bài kiểu này là nền tảng rất tốt cho các lab hồi quy Bayesian về sau, nơi công thức chuẩn liên hợp được mở rộng lên trường hợp nhiều tham số.


### Công thức cốt lõi và cách đọc

Công thức đề bài gợi ý cần thêm phương sai đã biết của likelihood $\sigma_2^2$, nhưng đề chưa ghi rõ giá trị này. Để solution khép kín và nhất quán với các bài liên hợp chuẩn trong repo, ở đây ta dùng giả định:

$$
Y\mid X=x \sim N(x, 1).
$$

Nếu prior là $X\sim N(\mu_0,\sigma_0^2)$ và có $n$ quan sát $y_1,\ldots,y_n$, thì posterior là:

$$
\frac{1}{\sigma_n^2} = \frac{1}{\sigma_0^2} + \frac{n}{\sigma_2^2},
\qquad
\frac{\mu_n}{\sigma_n^2} = \frac{\mu_0}{\sigma_0^2} + \frac{\sum_{i=1}^n y_i}{\sigma_2^2}.
$$

Ở đây:

- $\mu_0 = 16$,
- $\sigma_0^2 = 0.5$,
- $\sigma_2^2 = 1$,
- $n = 15$.


In [ ]:
observations = np.array([
    16.09, 15.99, 15.77, 16.04, 16.18,
    15.72, 15.98, 16.39, 15.51, 16.22,
    15.96, 15.81, 15.86, 16.46, 15.85
])

mu0 = 16.0
sigma0_sq = 0.5
sigma2_sq = 1.0
n = len(observations)

posterior_var_bai3 = 1 / (1 / sigma0_sq + n / sigma2_sq)
posterior_mean_bai3 = posterior_var_bai3 * (mu0 / sigma0_sq + observations.sum() / sigma2_sq)

print(f"Bai 3: n = {n}, tong mau = {observations.sum():.4f}, trung binh mau = {observations.mean():.6f}")
print(f"Bai 3: posterior = N({posterior_mean_bai3:.6f}, {posterior_var_bai3:.6f})")

x_grid = np.linspace(15.0, 17.0, 500)
prior_pdf = stats.norm.pdf(x_grid, loc=mu0, scale=np.sqrt(sigma0_sq))
posterior_pdf = stats.norm.pdf(x_grid, loc=posterior_mean_bai3, scale=np.sqrt(posterior_var_bai3))

plt.plot(x_grid, prior_pdf, label='Prior N(16, 0.5)')
plt.plot(x_grid, posterior_pdf, label=f'Posterior N({posterior_mean_bai3:.3f}, {posterior_var_bai3:.3f})')
plt.title('Bai 3 - Prior va posterior')
plt.xlabel('x')
plt.ylabel('density')
plt.legend()
plt.show()


### Interpretation

Với giả định $\sigma_2^2 = 1$, posterior thu được là

$$
X\mid y \sim N(15.99, 0.0588) \quad \text{(xấp xỉ)}.
$$

Ta thấy posterior mean nằm rất gần trung bình mẫu, còn posterior variance nhỏ hơn prior variance khá nhiều. Điều đó phản ánh đúng tinh thần Bayesian: dữ liệu đã kéo niềm tin ban đầu về vùng được quan sát và đồng thời làm cho độ bất định giảm xuống.


## 5. Bài 4 - Tiên nghiệm hỗn hợp từ ba phân phối Beta

**Đề bài.** Gọi $a,b$ lần lượt là chữ số gần cuối và chữ số cuối của MSSV. Ba nhóm chuyên gia đề xuất ba prior Beta khác nhau, rồi gộp lại thành một tiên nghiệm hỗn hợp với trọng số $40\%$, $35\%$, $25\%$. Hãy tính kỳ vọng, phương sai của tiên nghiệm hỗn hợp và vẽ ba thành phần cùng mật độ hỗn hợp.

### Ý nghĩa của bài tập

Đây là một bài rất hay vì nó cho thấy prior không nhất thiết phải là một phân phối đơn lẻ. Khi có nhiều nguồn chuyên gia hay nhiều nhóm tham chiếu khác nhau, ta hoàn toàn có thể mô hình hóa prior như một mixture distribution để phản ánh sự không đồng nhất ngay từ đầu.

Bài này cũng giúp sinh viên phân biệt rõ giữa **prior**, **mixture prior**, và **posterior**. Trong đề hiện tại, phần dữ liệu 67 sinh viên với 31 sinh viên đã đi làm chỉ là bối cảnh bổ sung; câu hỏi thực tế dừng ở việc phân tích tiên nghiệm hỗn hợp, chưa cập nhật sang posterior.


### Công thức cốt lõi và cách đọc

Với MSSV hiện tại, ta có:

$$
B_1 = \text{Beta}(35, a+b),
\qquad
B_2 = \text{Beta}(24, ab),
\qquad
B_3 = \text{Beta}(40, a+b+ab).
$$

Tiên nghiệm hỗn hợp là:

$$
\pi(p) = 0.4 f_{B_1}(p) + 0.35 f_{B_2}(p) + 0.25 f_{B_3}(p).
$$

Nếu thành phần thứ $i$ có kỳ vọng $\mu_i$ và phương sai $v_i$, thì:

$$
E[p] = \sum_i w_i\mu_i,
$$

và

$$
\mathrm{Var}(p) = \sum_i w_i(v_i + \mu_i^2) - E[p]^2.
$$

Với phân phối Beta$(\alpha,\beta)$, ta dùng:

$$
E[p] = \frac{\alpha}{\alpha+\beta},
\qquad
\mathrm{Var}(p) = \frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}.
$$


In [ ]:
a_bai4 = second_last_digit
b_bai4 = last_digit
weights = np.array([0.4, 0.35, 0.25])
beta_params = {
    'B1': (35, a_bai4 + b_bai4),
    'B2': (24, a_bai4 * b_bai4),
    'B3': (40, a_bai4 + b_bai4 + a_bai4 * b_bai4),
}

component_means = {}
component_vars = {}
for name, (alpha, beta) in beta_params.items():
    mean = alpha / (alpha + beta)
    var = alpha * beta / ((alpha + beta)**2 * (alpha + beta + 1))
    component_means[name] = mean
    component_vars[name] = var

ordered_names = ['B1', 'B2', 'B3']
means = np.array([component_means[name] for name in ordered_names])
variances = np.array([component_vars[name] for name in ordered_names])

mixture_mean_bai4 = np.dot(weights, means)
mixture_second_moment = np.dot(weights, variances + means**2)
mixture_var_bai4 = mixture_second_moment - mixture_mean_bai4**2

print(f"Bai 4: tham so Beta = {beta_params}")
print(f"Bai 4: ky vong tien nghiem hon hop = {mixture_mean_bai4:.6f}")
print(f"Bai 4: phuong sai tien nghiem hon hop = {mixture_var_bai4:.6f}")

grid_bai4 = np.linspace(0.001, 0.999, 500)
component_colors = {'B1': '#1f77b4', 'B2': '#ff7f0e', 'B3': '#2ca02c'}

mixture_density = np.zeros_like(grid_bai4)
for weight, name in zip(weights, ordered_names):
    alpha, beta = beta_params[name]
    density = stats.beta.pdf(grid_bai4, alpha, beta)
    mixture_density += weight * density
    plt.plot(grid_bai4, density, label=f'{name} = Beta({alpha}, {beta})', color=component_colors[name], alpha=0.85)

plt.plot(grid_bai4, mixture_density, color='black', linewidth=2.5, label='Prior hon hop')
plt.title('Bai 4 - Ba thanh phan Beta va tien nghiem hon hop')
plt.xlabel('p')
plt.ylabel('density')
plt.legend()
plt.show()


### Interpretation

Với MSSV minh họa hiện tại, tiên nghiệm hỗn hợp có kỳ vọng khoảng $0.7784$ và phương sai khoảng $0.00534$. Kỳ vọng này nằm giữa ba kỳ vọng thành phần, còn phương sai hỗn hợp không chỉ phụ thuộc vào độ phân tán bên trong từng Beta mà còn phụ thuộc vào độ chênh lệch giữa các thành phần với nhau.

Đồ thị cho thấy prior hỗn hợp là một sự pha trộn có trọng số của ba đường Beta ban đầu, chứ không phải chỉ là “lấy trung bình tham số rồi thay vào một Beta mới”.


## 6. Tổng kết nhanh

Bốn bài của đề đi qua bốn tầng ý tưởng khá điển hình:

- **Bài 1:** ghép các xác suất điều kiện bằng quy tắc xác suất toàn phần;
- **Bài 2:** xây posterior và lấy MAP;
- **Bài 3:** dùng liên hợp chuẩn để viết posterior Gaussian;
- **Bài 4:** mô tả prior như một mixture distribution.

Vì vậy dù đề chỉ dài bốn câu, nó vẫn là một bản ôn tập nhỏ nhưng khá sát tinh thần của cả chuỗi lab đầu.


In [ ]:
print('Tom tat nhanh voi MSSV hien tai')
print('-' * 50)
print(f"MSSV: {student_id}")
print(f"Bai 1: P(mat tat ve mat) = {100*p_eye:.2f}%")
print(f"Bai 2: y = {y_obs:.6f}, MAP = {map_estimate_bai2:.6f}")
print(f"Bai 3: posterior = N({posterior_mean_bai3:.6f}, {posterior_var_bai3:.6f})  (gia dinh sigma_2^2 = 1)")
print(f"Bai 4: E[p] = {mixture_mean_bai4:.6f}, Var(p) = {mixture_var_bai4:.6f}")
